# practicing prompt engineering with openai api or hugging face transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a baseline classifier with scikit-learn
- Evaluate using accuracy + confusion matrix
- Show how to encode categorical features

## 🔗 Prerequisites

- ✅ Python basics
- ✅ Jupyter Notebook basics

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 2**:
- practicing prompt engineering with openai api or hugging face transformers
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md`

---


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# small synthetic dataset
rng = np.random.default_rng(123)
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
color = rng.choice(['red','green','blue'], size=n)

y = ((x1 + 0.8*x2 + (color == 'red')*0.6 + rng.normal(scale=0.5, size=n)) > 0.2).astype(int)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'color': color, 'y': y})
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
 ('cat', OneHotEncoder(handle_unknown='ignore'), ['color']),
], remainder='passthrough')

clf = Pipeline([
 ('pre', pre),
 ('model', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print('confusion matrix:', confusion_matrix(y_test, y_pred))
print('\nreport:', classification_report(y_test, y_pred))


## 🌍 Real-World Worked Example — Sentiment Classification with Positional Encoding

**Industry context:**
- Twitter/X classifies 500M tweets/day using transformer-based sentiment models
- Amazon uses BERT-based models to classify product reviews automatically
- Customer service chatbots (Salesforce Einstein, Zendesk) use transformers for intent detection

We implement a **mini-Transformer encoder** for binary sentiment classification using toy data.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)
VOCAB_SIZE, SEQ_LEN, D_MODEL = 100, 10, 32

# ── Synthetic sentiment data ────────────────────────────────────────────────
# Positive: sequences with mostly high token IDs; Negative: low token IDs
def make_data(n=500):
    X, y = [], []
    for _ in range(n):
        label = np.random.randint(2)
        if label == 1:
            seq = np.random.randint(50, 100, SEQ_LEN)
        else:
            seq = np.random.randint(0, 50, SEQ_LEN)
        X.append(seq); y.append(label)
    return torch.tensor(np.array(X)), torch.tensor(y, dtype=torch.long)

X, y = make_data(800)
X_tr, y_tr = X[:640], y[:640]
X_te, y_te = X[640:], y[640:]

# ── Mini Transformer ────────────────────────────────────────────────────────
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
        # Learnable positional encoding
        self.pos_enc = nn.Embedding(SEQ_LEN, D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=4, dim_feedforward=64, batch_first=True)
        self.encoder  = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(D_MODEL, 2)
    def forward(self, x):
        pos = torch.arange(x.size(1)).unsqueeze(0)
        out = self.embed(x) + self.pos_enc(pos)
        out = self.encoder(out)
        return self.classifier(out.mean(1))  # pool over sequence

model   = MiniTransformer()
opt     = optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.CrossEntropyLoss()
losses, accs = [], []

for epoch in range(80):
    model.train()
    loss = loss_fn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    acc = (model(X_te).argmax(1)==y_te).float().mean().item()
    accs.append(acc)
print(f"Sentiment classification accuracy: {acc*100:.1f}%")
print("Real-world: BERT achieves 94%+ on SST-2 sentiment benchmark.")

plt.plot(losses); plt.title("Transformer Training Loss — Sentiment Classifier")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.tight_layout(); plt.show()

## 📚 References & Further Reading

**Foundational Paper:**
- Vaswani et al. (2017) — [Attention Is All You Need](https://arxiv.org/abs/1706.03762) *(must-read)*

**Follow-up:**
- Devlin et al. (2018) — [BERT](https://arxiv.org/abs/1810.04805)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)
- Touvron et al. (2023) — [LLaMA](https://arxiv.org/abs/2302.13971)

**Interactive:** [The Illustrated Transformer by Jay Alammar](https://jalammar.github.io/illustrated-transformer/)

**State-of-the-Art:** GPT-4 uses transformer architecture with ~1 trillion parameters. LLaMA 3.1 (70B) runs on a single server.

## 📝 Summary

In this notebook you studied **Practicing Prompt Engineering With Openai Api Or Hugging Face Transformers** — a key component of modern AI systems. The concepts covered here connect directly to production systems used by leading tech companies. Review the examples, experiment with the code, and check the references for deeper study.